# Tono · portero de calidad aprendido, y su auditoría de sesgo

> ⚠️ **NO ES UNA HERRAMIENTA DIAGNÓSTICA.** Proyecto educativo y experimental.

El portero basado en reglas fotográficas **fracasó**: ninguna de sus once
medidas predice el juicio de un dermatólogo mejor que el azar (AUC 0,547
combinándolas todas). La adecuación diagnóstica es semántica, no un
estadístico de píxeles. Aquí se sustituye por un modelo entrenado contra las
etiquetas de SCIN.

## La hipótesis incómoda que hay que medir

En SCIN, la tasa de «no evaluable» **según el dermatólogo** sube del 20,6% en
piel clara al 66,7% en piel oscura: **la etiqueta con la que se va a entrenar
ya está sesgada por tono de piel.**

Un modelo entrenado sobre eso puede aprender el atajo «piel oscura → rechazar»,
y sería un desastre silencioso: excluiría a un grupo antes de que ningún
clasificador de lesiones llegue a opinar. El portero por reglas no
discriminaba, pero tampoco funcionaba. Cambiar «no funciona pero es justo» por
«funciona pero discrimina» sería un mal negocio.

**La métrica que decide este experimento no es el AUROC, es la FPR por tono de
piel**: cuántas fotos *buenas* descarta en cada grupo.

In [ ]:
import subprocess, sys, os, time, glob, json
T0 = time.time()

for nombre, url in [('tono', 'https://github.com/GGGuardin/tono.git'),
                    ('cxr', 'https://github.com/GGGuardin/chest-xray-pneumonia.git')]:
    subprocess.run(['rm', '-rf', '/tmp/' + nombre], check=False)
    subprocess.run(['git', 'clone', '--depth', '1', '-q', url, '/tmp/' + nombre], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'albumentations'], check=True)

import torch
cap = torch.cuda.get_device_capability(0)
assert 'sm_%d%d' % cap in torch.cuda.get_arch_list(), 'GPU no soportada, relanza con T4'
torch.zeros(8, device='cuda').sum().item()
print('GPU:', torch.cuda.get_device_name(0), '| CUDA OK')

## 1. Descarga de SCIN

Bucket público de Google, sin registro. Descarga en paralelo con reescalado a
512 px y JPEG: de unos 5 GB a unos 300 MB.

In [ ]:
OUT = '/kaggle/working'
os.chdir('/tmp/tono')

!python -m datos.scin_descarga --out-dir /tmp/scin_imagenes --metadatos /tmp/.scin --hilos 24 --manifiesto {OUT}/manifiesto_scin.csv

In [ ]:
import pandas as pd

df = pd.read_csv(OUT + '/manifiesto_scin.csv')
assert (df.groupby('patient_id')['split'].nunique() == 1).all(), 'FUGA DE DATOS'
print('OK: ningun caso en mas de un split')
print()
print('Etiqueta (1 = NO evaluable) por tono de piel:')
tabla = df.groupby('fitzpatrick').agg(n=('label', 'size'), no_evaluables=('label', 'mean'))
print(tabla.round(3).to_string())
print()
print('Esa columna es el sesgo que ya viene en la etiqueta humana.')
print('Si el modelo lo copia, se vera en la FPR por tono al final.')

## 2. Entrenamiento

EfficientNet-B0 en lugar de DenseNet-121: la Fase 3 es llevarlo al navegador,
y cuantizada ronda los 5 MB frente a 28 MB. Mejor elegir ya el modelo que se va
a desplegar que entrenar dos veces.

In [ ]:
os.chdir('/tmp/cxr')
!python -m src.train --config /tmp/tono/configs/scin_portero.yaml --manifest {OUT}/manifiesto_scin.csv --out-dir {OUT}/runs/portero_scin

In [ ]:
h = pd.read_csv(OUT + '/runs/portero_scin/history.csv')
print(h.to_string(index=False))
print()
print('Referencia a batir: el portero por reglas daba AUC 0,547, o sea azar.')

## 3. Evaluación en test

In [ ]:
!python -m src.evaluate --checkpoint {OUT}/runs/portero_scin/best.pth --manifest {OUT}/manifiesto_scin.csv --split test --out-dir {OUT}/reports/test --n-boot 1000

## 4. La pregunta que decide el experimento

**FPR por tono de piel** = fotos buenas descartadas en cada grupo. Si sube con
la oscuridad de la piel, el modelo copió el sesgo de la etiqueta y **no se debe
desplegar**, por muy alto que sea su AUROC.

In [ ]:
!python -m src.fairness --predictions {OUT}/reports/test/predictions.csv --out-dir {OUT}/reports/equidad --attributes fitzpatrick,view --intersect "" --min-n 20

In [ ]:
sys.path.insert(0, '/tmp/tono/kaggle/portero')
from veredicto import evaluar_sesgo

sub = pd.read_csv(OUT + '/reports/equidad/subgrupos.csv')
resultado = evaluar_sesgo(sub)
columnas = ['subgrupo', 'n', 'fpr', 'fnr_infradiagnostico', 'auroc']

print('FPR = fotos BUENAS descartadas, por tono de piel')
print(resultado['tabla'][columnas].to_string(index=False))
print()
for clave in ('correlacion_tono_fpr', 'brecha_fpr'):
    if clave in resultado:
        print(clave, '=', resultado[clave])
veredicto = resultado['veredicto']
print()
print('>>> ' + veredicto)

## 5. Resumen

In [ ]:
resumen = {'objetivo': 'predecir que un dermatologo NO podria evaluar la foto',
           'referencia_reglas_auc': 0.547,
           'veredicto_equidad': veredicto,
           'fpr_por_tono': resultado.get('fpr_por_tono'),
           'correlacion_tono_fpr': resultado.get('correlacion_tono_fpr'),
           'brecha_fpr': resultado.get('brecha_fpr')}

for nombre, ruta in [('test', OUT + '/reports/test/metrics.json'),
                     ('equidad', OUT + '/reports/equidad/fairness.json')]:
    if os.path.exists(ruta):
        d = json.load(open(ruta))
        resumen[nombre] = {k: v for k, v in d.items() if k != 'detalle'}

resumen['minutos'] = round((time.time() - T0) / 60, 1)
json.dump(resumen, open(OUT + '/resumen.json', 'w'), indent=2, ensure_ascii=False)

auc = resumen.get('test', {}).get('auroc')
if auc:
    print('AUROC del portero aprendido: %.4f   (reglas: 0.547)' % auc)
print(json.dumps(resumen, indent=2, ensure_ascii=False)[:2500])

import shutil
for f_ in glob.glob(OUT + '/manifiesto_*.csv'):
    shutil.move(f_, '/tmp/' + os.path.basename(f_))